<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation
Install the libraries needed to fetch market data, compute rolling windows, and render charts in this notebook.

In [ ]:
!pip install yfinance pandas matplotlib

yfinance pulls prices, pandas provides rolling operations on DataFrames, and matplotlib underpins pandas plotting so .plot() works without extra imports. Installing them up front reduces environment drift and makes the notebook reproducible across machines.

## Imports and setup
We use yfinance to download historical prices from Yahoo Finance; the returned pandas objects expose rolling calculations and plotting backed by matplotlib.

In [ ]:
import yfinance as yf

Keeping imports minimal helps isolate the rolling-window logic we want to learn without hiding behavior behind frameworks. Using a single data source also reduces version mismatches, which is important when you validate signals and their alignment later.

## Fetch historical prices for testing
Pull a clean daily price history for NFLX over a fixed window so our rolling features have enough bars to stabilize and so results are reproducible.

In [ ]:
data = yf.download("NFLX", start="2020-01-01", end="2022-06-30")

yfinance returns a pandas DataFrame with OHLCV columns; we will focus on Close for simplicity. Many production workflows prefer Adjusted Close to include corporate actions, but the rolling mechanics are identical. Working with one asset lets us focus on window alignment before scaling out.

## Compute rolling features without lookahead
Define a window function that scores the latest price relative to its recent history, then apply it over a 30-bar rolling window; we keep the window uncentered so it only uses information available up to the current bar.

In [ ]:
def z_score(chunk):
    return (chunk.iloc[-1] - chunk.mean()) / chunk.std()

In [ ]:
rolled = data.Close.rolling(window=30).apply(z_score)

This yields a standardized feature where larger positive values mean the price sits above its recent mean by more local volatility. For backtests, join this feature to returns with a one-bar lag to avoid lookahead bias; we compute on bar t but must act on bar t+1. .apply makes the per-window logic explicit for learning, while vectorized rolling means/std are faster in production.

Compute the worst daily percentage change observed in each 30-day window to monitor downside clustering and inform risk sizing.

In [ ]:
min_pct_change = (
    data
    .Close
    .pct_change()
    .rolling(window=30)
    .min()
)

A rolling minimum of returns flags stress periods where losses accumulate and can guide position caps or de-risking rules. We compute returns first and then roll so the window advances strictly forward in time. As with the z-score, plan to lag this feature one bar before using it to drive decisions.

## Visualize and sanity-check feature distributions
Plot time series and histograms of both features to spot alignment issues and understand distribution shape before you backtest thresholds.

In [ ]:
rolled.plot()
rolled.hist(bins=20)
min_pct_change.plot()
min_pct_change.hist(bins=20)

Quick visuals reveal problems like unexpected spikes or long runs of NaNs; initial values should be NaN until 30 observations accumulate, which pandas handles automatically. Check that z-scores mostly sit in a sensible range (often within a few standard deviations) and that worst-return windows look plausible for the asset. Validating these basics now prevents quiet data leakage and miscalibrated signals later.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.